# Haz que el lago resbale: solución

Una solución de la tarea de práctica *Haz que el lago resbale*, en el [README](../README.md#práctica) de esta carpeta: intenta la tarea antes de leerla. El código reproduce el del notebook [Q-learning en FrozenLake](../frozen_lake.ipynb), con una semilla para cada ejecución para que los números se puedan reproducir.

## El lago resbaladizo

In [ ]:
import gymnasium as gym

env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
transitions = env.unwrapped.P
transitions[0][2]

[(0.33333333333333337, 4, 0, False),
 (0.3333333333333333, 1, 0, False),
 (0.33333333333333337, 0, 0, False)]

`env.unwrapped.P[state][action]` lista los resultados de una acción como `(probabilidad, estado siguiente, recompensa, terminado)`. DERECHA desde el inicio va a la derecha, al estado 1, una de cada tres veces; el resto resbala en perpendicular a la dirección prevista: ABAJO al estado 4, o ARRIBA contra la pared, lo que deja al agente en el inicio. Aquí se conoce la tabla entera, lo que nos permite calcular exactamente la mejor política posible: algo que un agente que aprende de la experiencia no puede hacer.

## Pregunta 1: con qué frecuencia, y cuánto varía

El entrenamiento del notebook, como una función, con una semilla para cada ejecución:

In [ ]:
import random

import numpy as np


def train(learning_rate, gamma, n_training_episodes, seed, max_steps=99, max_epsilon=1.0, min_epsilon=0.05,
          decay_rate=0.0005):
    random.seed(seed)
    env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
    env.reset(seed=seed)
    env.action_space.seed(seed)
    qtable = np.zeros((env.observation_space.n, env.action_space.n))
    for episode in range(n_training_episodes):
        epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)
        state, info = env.reset()
        for step in range(max_steps):
            if random.uniform(0, 1) > epsilon:
                action = np.argmax(qtable[state])
            else:
                action = env.action_space.sample()
            new_state, reward, terminated, truncated, info = env.step(action)
            qtable[state][action] += learning_rate * (
                reward + gamma * np.max(qtable[new_state]) - qtable[state][action]
            )
            if terminated or truncated:
                break
            state = new_state
    return qtable

Su evaluación, con la política voraz, en un entorno propio, para que todas las tablas Q se evalúen con la misma secuencia de resbalones:

In [ ]:
def success_rate(qtable, n_eval_episodes=1000, seed=1000, max_steps=99):
    env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
    env.reset(seed=seed)
    successes = 0
    for episode in range(n_eval_episodes):
        state, info = env.reset()
        for step in range(max_steps):
            state, reward, terminated, truncated, info = env.step(np.argmax(qtable[state]))
            if terminated or truncated:
                successes += reward
                break
    return successes / n_eval_episodes

Cinco ejecuciones con la configuración del notebook: α 0,7, γ 0,95, 10.000 episodios.

In [ ]:
seeds = range(5)
notebook_tables = [train(0.7, 0.95, 10_000, seed) for seed in seeds]
[success_rate(qtable) for qtable in notebook_tables]

[0.729, 0.309, 0.718, 0.379, 0.619]

Entre 0,31 y 0,73, con el mismo código y la misma configuración: en un lago resbaladizo, una ejecución dice poco.

La evaluación añade su propio ruido. La primera de esas tablas Q, evaluada cinco veces con 100 episodios, como hace el notebook:

In [ ]:
[success_rate(notebook_tables[0], n_eval_episodes=100, seed=evaluation_seed) for evaluation_seed in range(5)]

[0.63, 0.75, 0.71, 0.82, 0.79]

La misma tabla Q puntúa entre 0,63 y 0,82, alrededor del 0,729 que miden 1.000 episodios. Con $n$ episodios, una tasa de éxito cercana a 0,7 se estima con un margen de aproximadamente $\pm 1{,}96 \sqrt{0{,}7 \cdot 0{,}3 / n}$ (95 %): ±9 puntos con 100 episodios, ±3 con 1.000, y por eso la tarea pide 1.000. La dispersión entre las ejecuciones de arriba, de 0,31 a 0,73, es mucho mayor que ±3 puntos: viene del entrenamiento, no de la evaluación.

## El techo

Como se conoce la tabla de transiciones, la **iteración de valores** (*value iteration*) calcula los valores de acción óptimos $Q^*$: aplica la ecuación de Bellman del [notebook 2](../bellman_equation.ipynb) como una asignación, a cada estado y acción, hasta que los valores dejan de cambiar. En un lago resbaladizo, su lado derecho es un promedio sobre los tres resultados de cada acción.

In [ ]:
def action_values(state_values, gamma):
    q = np.zeros((16, 4))
    for state in range(16):
        for action in range(4):
            for probability, next_state, reward, terminated in transitions[state][action]:
                q[state, action] += probability * (reward + gamma * state_values[next_state] * (not terminated))
    return q


def optimal_action_values(gamma, iterations=2000):
    state_values = np.zeros(16)
    for _ in range(iterations):
        state_values = action_values(state_values, gamma).max(axis=1)
    return action_values(state_values, gamma)


optimal_values = {gamma: optimal_action_values(gamma) for gamma in [0.95, 0.99]}

La tasa de éxito de una política dentro del límite de 99 pasos, calculada exactamente: la probabilidad de llegar a la meta en como mucho $n$ pasos, construida paso a paso desde $n = 0$.

In [ ]:
def success_probability(policy, max_steps=99):
    success_within = np.zeros(16)
    for _ in range(max_steps):
        success_within_one_more = np.zeros(16)
        for state in range(16):
            for probability, next_state, reward, terminated in transitions[state][policy[state]]:
                success_within_one_more[state] += probability * (
                    reward + success_within[next_state] * (not terminated)
                )
        success_within = success_within_one_more
    return success_within[0]


optimal_policies = {gamma: q.argmax(axis=1) for gamma, q in optimal_values.items()}
{gamma: round(success_probability(policy), 3) for gamma, policy in optimal_policies.items()}

{0.95: np.float64(0.728), 0.99: np.float64(0.738)}

Ningún agente puede superar 0,738 en 99 pasos: ese es el 74 % de la tarea. Las políticas óptimas de los dos valores de γ solo se diferencian en el estado 2, en la fila de arriba: γ 0,95 elige IZQUIERDA, que resbala ABAJO al estado 6 una de cada tres veces, un camino más rápido para bajar; γ 0,99 elige ARRIBA, que se queda en la fila de arriba. El descuento hace que una ruta lenta y segura valga menos, así que γ 0,95 acepta algo más de riesgo (0,728).

Sin el límite de pasos, la política de γ 0,99 lo haría mejor:

In [ ]:
round(success_probability(optimal_policies[0.99], max_steps=10_000), 3)

np.float64(0.824)

82,4 %: casi 9 puntos se pierden solo por el límite de 99 pasos, porque la política segura gasta muchos movimientos chocando con las paredes, y algunos episodios se quedan sin pasos. El otro 18 % cae en agujeros, y ninguna política puede evitarlo. Las acciones que no pueden caer en un agujero, estado por estado, muestran por qué:

In [ ]:
action_names = ["LEFT", "DOWN", "RIGHT", "UP"]
holes = {5, 7, 11, 12}
goal = 15

{
    state: [
        action_names[action]
        for action in range(4)
        if all(next_state not in holes for _, next_state, _, _ in transitions[state][action])
    ]
    for state in range(16)
    if state not in holes and state != goal
}

{0: ['LEFT', 'DOWN', 'RIGHT', 'UP'],
 1: ['UP'],
 2: ['LEFT', 'DOWN', 'RIGHT', 'UP'],
 3: ['UP'],
 4: ['LEFT'],
 6: [],
 8: ['UP'],
 9: ['DOWN'],
 10: ['LEFT'],
 13: ['RIGHT'],
 14: ['LEFT', 'DOWN', 'RIGHT', 'UP']}

El estado 6 no tiene ninguna acción segura: cada movimiento resbala al agujero 5 o al agujero 7 una de cada tres veces. La ruta segura baja por la columna de la izquierda (0, 4, 8, 9, 13, 14), pero en el estado 9 la única acción segura, ABAJO, resbala a la DERECHA al estado 10 una de cada tres veces, y desde el estado 10 la única acción segura, IZQUIERDA, resbala ARRIBA al estado 6 una de cada tres veces. Toda ruta a la meta pasa por el estado 9 o el 10, así que ninguna política puede tapar esa fuga. La fila de arriba no es un rodeo: desde ella, las únicas formas de bajar son el estado 4, de vuelta a la columna de la izquierda, el estado 6, o un agujero.

## Pregunta 2: qué configuraciones llegan al techo

Cinco ejecuciones de cada configuración: un cambio cada vez, y después los dos que importan juntos. Esta celda tarda unos dos minutos.

In [ ]:
import pandas as pd

settings = {
    "notebook (α 0.7, γ 0.95, 10,000 episodes)": (0.7, 0.95, 10_000),
    "α 0.1": (0.1, 0.95, 10_000),
    "γ 0.99": (0.7, 0.99, 10_000),
    "50,000 episodes": (0.7, 0.95, 50_000),
    "α 0.1 and γ 0.99": (0.1, 0.99, 10_000),
}
trained_tables = {
    name: [train(learning_rate, gamma, n_episodes, seed) for seed in seeds]
    for name, (learning_rate, gamma, n_episodes) in settings.items()
}
results = pd.DataFrame(
    {name: [success_rate(qtable) for qtable in tables] for name, tables in trained_tables.items()},
    index=[f"seed {seed}" for seed in seeds],
).T
results["mean"] = results.mean(axis=1)
results.round(3)

,seed 0,seed 1,seed 2,seed 3,seed 4,mean
"notebook (α 0.7, γ 0.95, 10,000 episodes)",0.729,0.309,0.718,0.379,0.619,0.551
α 0.1,0.721,0.297,0.729,0.722,0.721,0.638
γ 0.99,0.225,0.401,0.584,0.165,0.584,0.392
"50,000 episodes",0.703,0.220,0.729,0.584,0.601,0.567
α 0.1 and γ 0.99,0.728,0.718,0.718,0.728,0.728,0.724


- **Ningún cambio individual basta.** α 0,1 por sí solo llega al techo de γ 0,95 (0,728) en cuatro ejecuciones de cinco, pero la semilla 1 falla con 0,297 (ver *Un fallo de α 0,1*, más abajo). γ 0,99 por sí solo y cinco veces más episodios no ayudan: con α 0,7 las ejecuciones se reparten de 0,17 a 0,73, y con cinco ejecuciones las diferencias entre esas medias están dentro de esa dispersión, así que no son prueba de que ninguno de los cambios empeore las cosas.
- **α 0,1 y γ 0,99 juntos llegan a 0,72–0,73 en todas las ejecuciones:** el techo de 0,738, dentro de los ±3 puntos de una evaluación de 1.000 episodios.
- **Varias ejecuciones por configuración son parte de la respuesta.** A juzgar solo por la semilla 0, la configuración del notebook (0,729) parecería tan buena como las mejores.

### Por qué importa la tasa de aprendizaje

En el lago resbaladizo el objetivo de una actualización, $r + \gamma \max_{a'} Q(s', a')$, es aleatorio: depende de hacia dónde resbaló el agente. Con una tasa de aprendizaje constante, $Q(s, a)$ es una media ponderada exponencialmente de los objetivos que ha visto, con peso $\alpha (1 - \alpha)^k$ para el objetivo de hace $k$ actualizaciones. Si los objetivos fueran independientes, su varianza sería $\frac{\alpha}{2 - \alpha}$ veces la varianza de un objetivo: 0,54 con α 0,7, 0,053 con α 0,1. Con α 0,7 la estimación es básicamente los dos o tres últimos objetivos, por mucho que dure el entrenamiento, y por eso 50.000 episodios no ayudan. En el lago determinista todos los objetivos de una acción son iguales, así que allí α no importaba.

Lo que cuesta ese ruido depende del **margen** entre la mejor y la segunda mejor acción de cada estado: cuando el error es mayor que el margen, la elección voraz entre las dos es lanzar una moneda. Los márgenes de los valores óptimos en los estados donde decide el agente (el estado 6 se deja fuera: IZQUIERDA y DERECHA empatan ahí, imágenes especulares la una de la otra):

In [ ]:
decision_states = [state for state in range(16) if state not in holes and state not in (goal, 6)]


def margins(q):
    sorted_values = np.sort(q[decision_states], axis=1)
    return sorted_values[:, -1] - sorted_values[:, -2]


pd.DataFrame({f"γ {gamma}": margins(q) for gamma, q in optimal_values.items()}, index=decision_states).round(3)

,γ 0.95,γ 0.99
0,0.008,0.014
1,0.049,0.155
2,0.007,0.033
3,0.042,0.151
4,0.057,0.179
8,0.066,0.184
9,0.086,0.195
10,0.056,0.118
13,0.119,0.212
14,0.033,0.042


γ 0,99 tiene márgenes mayores en todos los estados (mediana de 0,153 frente a 0,052): un agente que valora más el futuro lejano pierde más al caer en un agujero, así que las acciones seguras y las arriesgadas se diferencian más. Los márgenes más pequeños, en los estados 0 y 2, son donde es más probable que un agente con ruido elija mal.

A continuación, el error de los valores aprendidos en relación con esos márgenes (para cada estado de decisión, el mayor error de sus cuatro acciones dividido entre su margen; la mediana sobre los estados y después sobre las cinco ejecuciones), y la proporción de estados de decisión en los que la acción voraz aprendida es la óptima (la media sobre las ejecuciones):

In [ ]:
def relative_error(qtable, gamma):
    errors = np.abs(qtable - optimal_values[gamma])[decision_states].max(axis=1)
    return np.median(errors / margins(optimal_values[gamma]))


def optimal_choices(qtable, gamma):
    learned_actions = qtable[decision_states].argmax(axis=1)
    return np.mean(learned_actions == optimal_values[gamma][decision_states].argmax(axis=1))


pd.DataFrame(
    {
        "error / margin": [
            np.median([relative_error(qtable, settings[name][1]) for qtable in tables])
            for name, tables in trained_tables.items()
        ],
        "optimal choices": [
            np.mean([optimal_choices(qtable, settings[name][1]) for qtable in tables])
            for name, tables in trained_tables.items()
        ],
    },
    index=list(trained_tables),
).round(2)

,error / margin,optimal choices
"notebook (α 0.7, γ 0.95, 10,000 episodes)",2.96,0.84
α 0.1,0.90,0.88
γ 0.99,2.06,0.84
"50,000 episodes",2.81,0.80
α 0.1 and γ 0.99,0.62,1.00


Con α 0,7 los errores son de dos a tres veces los márgenes, sea cual sea γ o el número de episodios, y aproximadamente una decisión de cada seis es errónea. α 0,1 baja los errores hasta más o menos los márgenes de γ 0,95; con γ 0,99, cuyos márgenes son mayores, quedan por debajo (0,62), y el agente elige la acción óptima en todos los estados de decisión de todas las ejecuciones. Así que γ 0,99 ayuda de dos formas: su política óptima es algo más segura (0,738 frente a 0,728), y sus márgenes mayores son más fáciles de aprender. Ninguna de las dos se nota mientras domina el ruido de α 0,7.

### Un fallo de α 0,1

La ejecución fallida de α 0,1 (semilla 1), frente a los valores óptimos, en los estados donde su elección voraz es distinta:

In [ ]:
failed_run = trained_tables["α 0.1"][1]
wrong_states = [
    state for state in decision_states if failed_run[state].argmax() != optimal_values[0.95][state].argmax()
]
pd.concat(
    {
        "learned": pd.DataFrame(failed_run[wrong_states], index=wrong_states, columns=action_names),
        "optimal": pd.DataFrame(optimal_values[0.95][wrong_states], index=wrong_states, columns=action_names),
    },
    axis=1,
).round(3)

learned                      optimal                     
      LEFT   DOWN  RIGHT     UP    LEFT   DOWN  RIGHT     UP
2    0.128  0.132  0.125  0.117   0.153  0.147  0.146  0.140
9    0.291  0.299  0.328  0.214   0.247  0.375  0.289  0.213
14   0.502  0.580  0.669  0.579   0.518  0.724  0.690  0.622

Su error caro está en el estado 9, en la ruta segura: DERECHA resbala ARRIBA al agujero 5 una de cada tres veces, mientras que ABAJO no puede caer. Sus estimaciones de las mejores acciones son demasiado bajas (ABAJO en el estado 9: 0,299 frente a 0,375; ABAJO en el estado 14: 0,580 frente a 0,724), lo que sugiere un entrenamiento que no ha terminado. Si es así, más episodios deberían arreglarlo:

In [ ]:
success_rate(train(0.1, 0.95, 50_000, seed=1))

0.514

No lo arreglan: 0,514. La tabla de errores lo explica mejor. Con α 0,1 y γ 0,95 los errores son más o menos tan grandes como los márgenes (0,90), así que aproximadamente una decisión de cada ocho es errónea, y en esta ejecución una de ellas cae en el estado 9. Un α constante mantiene el mismo nivel de ruido por mucho que entrene, así que más episodios dan una ejecución distinta, no una con menos ruido. Con γ 0,99 el margen en el estado 9 se duplica con creces (0,195 frente a 0,086), y α 0,1 pasa a ser suficiente. La alternativa de los libros de texto, que no se prueba aquí, es una tasa de aprendizaje que disminuye con el número de visitas a cada estado y acción: entonces el Q-learning converge a $Q^*$ (Watkins y Dayan, 1992), a cambio de aprender más despacio.

## Pregunta 3: la acción del inicio

In [ ]:
pd.Series(optimal_values[0.99][0], index=action_names).round(3)

LEFT     0.542
DOWN     0.528
RIGHT    0.528
UP       0.522
dtype: float64

Todas las acciones del inicio mezclan los mismos tres resultados: quedarse en la esquina (un choque contra una pared), el estado 4 bajando por la columna de la izquierda, o el estado 1 en la fila de arriba. IZQUIERDA da quedarse, quedarse o estado 4; ABAJO y DERECHA dan quedarse, estado 4 o estado 1, la misma mezcla y por tanto el mismo valor; ARRIBA da quedarse, quedarse o estado 1. Los valores solo ordenan cuánto prefiere cada acción el estado 4 al estado 1. El estado 4 vale más porque la columna de la izquierda es la ruta segura, mientras que desde la fila de arriba el agente tiene que volver o bajar por el estado 6. IZQUIERDA, que apunta lejos de la meta, es la mejor porque las paredes convierten dos de sus tres resultados en una espera inofensiva, y el tercero es el bueno.

In [ ]:
{name: [action_names[qtable[0].argmax()] for qtable in tables] for name, tables in trained_tables.items()}

{'notebook (α 0.7, γ 0.95, 10,000 episodes)': ['LEFT',
  'LEFT',
  'LEFT',
  'LEFT',
  'LEFT'],
 'α 0.1': ['LEFT', 'LEFT', 'LEFT', 'LEFT', 'LEFT'],
 'γ 0.99': ['LEFT', 'RIGHT', 'LEFT', 'LEFT', 'LEFT'],
 '50,000 episodes': ['LEFT', 'RIGHT', 'LEFT', 'LEFT', 'LEFT'],
 'α 0.1 and γ 0.99': ['LEFT', 'LEFT', 'LEFT', 'LEFT', 'LEFT']}

Casi todas las ejecuciones encuentran IZQUIERDA, a pesar del margen más pequeño de todos (0,014); las dos que eligen DERECHA son ejecuciones de α 0,7. DERECHA cuesta poco aquí: vale 0,528 frente a 0,542.

## Qué dice una buena respuesta

- **Pregunta 1.** La tasa de éxito varía mucho entre ejecuciones (de 0,31 a 0,73 con la configuración del notebook), así que una ejecución no demuestra nada; 100 episodios de evaluación añaden ±9 puntos por su cuenta, 1.000 unos ±3.
- **Pregunta 2.** Ninguna política llega a la meta siempre: la ruta segura tiene fugas por los estados 9, 10 y 6, y el límite de pasos cuesta unos 9 puntos más, así que el objetivo es alrededor del 74 %. Lo que importa es la tasa de aprendizaje: los resbalones hacen que cada actualización sea una muestra aleatoria, que un α alto sigue en lugar de promediar, y más episodios no pueden arreglarlo. Con un α bajo, γ 0,99 llega entonces al techo en todas las ejecuciones. Una buena respuesta muestra varias ejecuciones por configuración, y cambia una cosa cada vez antes de combinarlas.
- **Pregunta 3.** IZQUIERDA, lejos de la meta: contra las paredes solo puede esperar o resbalar ABAJO, así que nunca manda al agente a la fila de arriba. Una buena respuesta lo explica con la regla de los resbalones, no como un error del agente.
- **Cuidado con** una tabla Q que no se volvió a crear entre ejecuciones (el entrenamiento continúa entonces desde la última), y con conclusiones sacadas de una sola ejecución o de 100 episodios de evaluación.